# 02. Correlación y retardos

Este cuaderno compara PM2.5 con mediciones pasadas y con el valor observado 24 horas después. Las gráficas ayudan a entender las variables de entrada del proyecto.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

carpeta = Path.cwd()
if not (carpeta / "data").exists():
    carpeta = carpeta.parent
limpios = pd.read_csv(carpeta / "data" / "datos limpios.csv", parse_dates=["fecha_hora"])
datos = pd.read_csv(carpeta / "data" / "datos_modelo.csv")
print(f"Filas con variables y objetivo: {len(datos)}")

## Autocorrelación

Cada valor compara mediciones separadas por el retardo indicado. Los pares con faltantes se omiten.

In [ ]:
serie = limpios.set_index("fecha_hora").pm25_original
autocorrelacion = pd.Series({k: serie.corr(serie.shift(k)) for k in range(1, 49)})
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(autocorrelacion.index, autocorrelacion.values, color="steelblue")
ax.axvline(1, color="darkorange", linestyle="--", label="1 hora")
ax.axvline(24, color="seagreen", linestyle="--", label="24 horas")
ax.set(title="Autocorrelación de PM2.5 hasta 48 horas", xlabel="Retardo (horas)", ylabel="Correlación")
ax.legend()
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()
print("Correlación en retardos 1 y 24:")
print(autocorrelacion.loc[[1, 24]].round(3))

## Correlación de Pearson

La matriz resume asociaciones lineales entre las entradas y el objetivo. Correlación no significa causalidad.

In [ ]:
columnas = ["pm25_t", "pm25_lag1", "pm25_lag24", "hora", "dia_semana", "target_24h"]
corr = datos[columnas].corr()
fig, ax = plt.subplots(figsize=(8, 6))
imagen = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(columnas)), columnas, rotation=35, ha="right")
ax.set_yticks(range(len(columnas)), columnas)
for i in range(len(columnas)):
    for j in range(len(columnas)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Correlación entre variables y objetivo")
fig.colorbar(imagen, ax=ax, label="Correlación")
plt.tight_layout()
plt.show()

## Medición actual y valor a 24 horas

Cada punto es un par observado. La diagonal muestra la regla de persistencia, que repite el valor actual.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(datos.pm25_t, datos.target_24h, alpha=0.25, s=16, color="slateblue")
limite = max(datos.pm25_t.max(), datos.target_24h.max())
ax.plot([0, limite], [0, limite], color="darkorange", linestyle="--", label="Persistencia (y = x)")
ax.set(title="PM2.5 en t y en t+24", xlabel="PM2.5 en t (µg/m³)", ylabel="PM2.5 en t+24 (µg/m³)")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()